# Parsing and Statistics with ThinkPack

This notebook demonstrates `thinkpack.parse` and `thinkpack.compute_stats` — the two core tools for analysing model outputs after generation.

No GPU or model download is required. The notebook uses `ModelInfo` directly, so it can run anywhere.

**What this notebook covers:**
1. Parsing single responses — the four reasoning outcomes
2. `ParsedResponse` fields — what each flag means
3. Batch parsing — flat and nested (tasks × samples) inputs
4. Token counting
5. `compute_stats` on a flat batch
6. `ResponseStats` fields and shorthand properties
7. Nested input and macro-averaging
8. Pass rates with `results`

In [ ]:
import sys

# add the src directory to the path so thinkpack can be imported without installation
sys.path.insert(0, "../../src")

from thinkpack import ModelInfo, compute_stats, parse

In [ ]:
# ModelInfo describes how a model wraps its reasoning block.
# prefixed=False means the model generates the opening tag itself (e.g. Qwen3).
# prefixed=True means the chat template injects the opening tag (e.g. OLMo-3).
# when working with a real tokenizer, use detect_model(tokenizer) instead.
model_info = ModelInfo(prefixed=False)

## 1. Parsing Single Responses

`parse()` splits a raw model output into a `ParsedResponse` containing the reasoning content, the answer, and a set of boolean flags describing the outcome.

There are four mutually exclusive reasoning outcomes — together they always sum to 1:

| Outcome | Description |
|---|---|
| **valid** | Block opened, closed, non-empty content |
| **empty** | Block opened and closed, but content was blank |
| **truncated** | Block opened but never closed (hit token limit) |
| **missing** | No reasoning block structure found at all |

In [ ]:
# outcome 1: valid — a complete, non-blank reasoning block
r = parse(
    response="<think>\nLet me work this out step by step.\n</think>\nThe answer is 42.",
    model_info=model_info,
)
print(f"reasoning : {r.reasoning!r}")
print(f"answer    : {r.answer!r}")
print(f"valid     : {r.has_valid_reasoning}")
print(f"invalid   : {r.has_invalid_reasoning}")

In [ ]:
# outcome 2: empty — block opened and closed but content is blank
r = parse(
    response="<think>\n</think>\nThe answer is 42.",
    model_info=model_info,
)
print(f"empty     : {r.has_empty_reasoning}")
print(f"valid     : {r.has_valid_reasoning}")
print(f"answer    : {r.answer!r}")

In [ ]:
# outcome 3: truncated — block opened but generation was cut off before the close tag
r = parse(
    response="<think>\nLet me work this out...",
    model_info=model_info,
)
print(f"truncated : {r.has_truncated_reasoning}")
print(f"reasoning : {r.reasoning!r}")
print(f"answer    : {r.answer!r}")

In [ ]:
# outcome 4: missing — no reasoning block found; the whole response is treated as the answer
r = parse(
    response="The answer is 42.",
    model_info=model_info,
)
print(f"missing   : {r.has_missing_reasoning}")
print(f"answer    : {r.answer!r}")

## 2. `ParsedResponse` Fields

Here is a summary of all fields on `ParsedResponse`:

| Field | Type | Description |
|---|---|---|
| `reasoning` | `str` | Content inside the reasoning block (empty string if no block) |
| `answer` | `str` | Text after the closing tag (or the full response if no block) |
| `reasoning_tag` | `str \| None` | Tag name detected, e.g. `"think"` (`None` if missing) |
| `has_valid_reasoning` | `bool` | Block complete and non-blank |
| `has_empty_reasoning` | `bool` | Block complete but blank |
| `has_truncated_reasoning` | `bool` | Block opened, never closed |
| `has_missing_reasoning` | `bool` | No block found at all |
| `has_invalid_reasoning` | `bool` | Property — `True` if not valid (any of the three invalid sub-types) |
| `has_answer` | `bool` | Property — `True` if `answer` is non-blank |
| `reasoning_token_count` | `int \| None` | Token length of reasoning (populated with `calculate_tokens=True`) |
| `answer_token_count` | `int \| None` | Token length of answer (populated with `calculate_tokens=True`) |

In [ ]:
# inspect all fields on a parsed response
r = parse(
    response="<think>\nSome reasoning.\n</think>\nThe answer.",
    model_info=model_info,
)
print(f"reasoning             : {r.reasoning!r}")
print(f"answer                : {r.answer!r}")
print(f"reasoning_tag         : {r.reasoning_tag!r}")
print(f"has_valid_reasoning   : {r.has_valid_reasoning}")
print(f"has_empty_reasoning   : {r.has_empty_reasoning}")
print(f"has_truncated_reasoning: {r.has_truncated_reasoning}")
print(f"has_missing_reasoning : {r.has_missing_reasoning}")
print(f"has_invalid_reasoning : {r.has_invalid_reasoning}")
print(f"has_answer            : {r.has_answer}")
print(f"reasoning_token_count : {r.reasoning_token_count}")

## 3. Batch Parsing

`parse()` accepts three input shapes:
- A single `str` → returns a single `ParsedResponse`
- A `list[str]` → returns a `list[ParsedResponse]`
- A `list[list[str]]` → returns a `list[list[ParsedResponse]]` (tasks × samples)

The nested shape is the natural output of vLLM with `n > 1` samples per prompt.

In [ ]:
# flat list: one response per item
responses = [
    "<think>\nreasoning A\n</think>\nanswer A",
    "<think>\n</think>\nanswer B",  # empty reasoning
    "<think>\ncut off...",  # truncated
    "plain answer",  # missing reasoning
]

parsed = parse(response=responses, model_info=model_info)

for i, r in enumerate(parsed):
    # show which outcome each response hit
    outcome = (
        "valid"
        if r.has_valid_reasoning
        else "empty"
        if r.has_empty_reasoning
        else "truncated"
        if r.has_truncated_reasoning
        else "missing"
    )
    print(f"[{i}] {outcome:10s}  answer={r.answer!r}")

In [ ]:
# nested list: [task][sample] — e.g. 2 tasks with 3 samples each
nested_responses = [
    [  # task 0 — three samples
        "<think>\nreasoning\n</think>\nanswer",
        "<think>\nreasoning\n</think>\nanswer",
        "plain answer",
    ],
    [  # task 1 — two samples
        "<think>\nreasoning\n</think>\nanswer",
        "<think>\ncut off...",
    ],
]

nested_parsed = parse(response=nested_responses, model_info=model_info)

for task_idx, task in enumerate(nested_parsed):
    valid = sum(r.has_valid_reasoning for r in task)
    print(f"task {task_idx}: {valid}/{len(task)} valid")

## 4. Token Counting

Pass `calculate_tokens=True` and a tokenizer to populate `reasoning_token_count` and `answer_token_count` on each `ParsedResponse`. This is useful for tracking how much of the model's token budget was spent on reasoning versus the answer.

In [ ]:
# token counting requires a real tokenizer — load one if available
try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", trust_remote_code=True)
    r = parse(
        response="<think>\nSome reasoning here.\n</think>\nThe answer.",
        tokenizer=tokenizer,
        calculate_tokens=True,
    )
    print(f"reasoning_token_count : {r.reasoning_token_count}")
    print(f"answer_token_count    : {r.answer_token_count}")
except Exception as e:
    # skip gracefully if no tokenizer is available in this environment
    print(f"skipped: {e}")

## 5. `compute_stats` on a Flat Batch

`compute_stats()` aggregates a list of `ParsedResponse` objects into a single `ResponseStats` with rates for each reasoning outcome. All rates are in `[0, 1]`.

In [ ]:
# parse a small flat batch with a mix of outcomes
responses = [
    "<think>\nreasoning\n</think>\nanswer",  # valid
    "<think>\nreasoning\n</think>\nanswer",  # valid
    "<think>\n</think>\nanswer",  # empty
    "<think>\ncut off...",  # truncated
    "plain answer",  # missing
    "plain answer",  # missing
]

parsed = parse(response=responses, model_info=model_info)
s = compute_stats(responses=parsed)

print(f"total              : {s.total}")
print(f"valid_reasoning    : {s.valid_reasoning_rate:.0%}")
print(f"invalid_reasoning  : {s.invalid_reasoning_rate:.0%}")
print(f"  missing          : {s.missing_reasoning_rate:.0%}")
print(f"  empty            : {s.empty_reasoning_rate:.0%}")
print(f"  truncated        : {s.truncated_reasoning_rate:.0%}")
print(f"answer             : {s.answer_rate:.0%}")

## 6. `ResponseStats` Fields and Shorthand Properties

Every rate field has a shorthand property for convenience in tight loops or interactive analysis:

| Property | Full name |
|---|---|
| `vr` | `valid_reasoning_rate` |
| `ir` | `invalid_reasoning_rate` |
| `mr` | `missing_reasoning_rate` |
| `tr` | `truncated_reasoning_rate` |
| `er` | `empty_reasoning_rate` |
| `ar` | `answer_rate` |

The invariants always hold:
- `vr + mr + tr + er == 1.0`
- `vr + ir == 1.0`

In [ ]:
# shorthands are convenient for quick inspection
print(f"vr={s.vr:.0%}  mr={s.mr:.0%}  tr={s.tr:.0%}  er={s.er:.0%}")

# verify invariants
print(f"vr+mr+tr+er == {s.vr + s.mr + s.tr + s.er:.1f}")
print(f"vr+ir       == {s.vr + s.invalid_reasoning_rate:.1f}")

## 7. Nested Input and Macro-Averaging

For `list[list[ParsedResponse]]` input (tasks × samples), `compute_stats` computes rates per task then **macro-averages** them — so each task contributes equally regardless of sample count. `total` is always the sum across all tasks.

Macro-averaging is the standard in evaluation benchmarks: a task with 10 samples does not outweigh one with 2.

In [ ]:
# two tasks with different numbers of samples
# task 0: 3 samples, 1 valid → vr = 1/3
# task 1: 1 sample,  1 valid → vr = 1/1
# macro-avg: (1/3 + 1.0) / 2 ≈ 0.667  (micro would be 2/4 = 0.5)
nested_responses = [
    [
        "<think>\nreasoning\n</think>\nanswer",
        "plain answer",
        "plain answer",
    ],
    [
        "<think>\nreasoning\n</think>\nanswer",
    ],
]

nested_parsed = parse(response=nested_responses, model_info=model_info)
s = compute_stats(responses=nested_parsed)

print(f"total : {s.total}")
print(f"vr    : {s.vr:.3f}  (macro-avg of 0.333 and 1.0)")

## 8. Pass Rates with `results`

Pass `results` (a matching list of `bool`) to compute correctness metrics:

| Field | Description |
|---|---|
| `pass_at_1` | Fraction of responses that answered correctly |
| `rpass_at_1` | Fraction correct **among responses with valid reasoning only** |

`rpass_at_1` isolates the quality of the model's reasoning — it answers: when the model actually thinks, does it get it right?

In [ ]:
# 4 responses: 2 valid-reasoning, 1 missing-reasoning, 1 truncated
responses = [
    "<think>\nreasoning\n</think>\nanswer",  # valid — correct
    "<think>\nreasoning\n</think>\nanswer",  # valid — wrong
    "plain answer",  # missing — correct
    "<think>\ncut off...",  # truncated — wrong
]
results = [True, False, True, False]

parsed = parse(response=responses, model_info=model_info)
s = compute_stats(responses=parsed, results=results)

# pass_at_1: 2 correct out of 4
print(f"pass_at_1  : {s.pass_at_1:.0%}")

# rpass_at_1: 1 correct out of the 2 valid-reasoning responses
print(f"rpass_at_1 : {s.rpass_at_1:.0%}")